# Deepfake detector - Colab training

**Before running:** Runtime -> Change runtime type -> T4 GPU.

**Before each session:** push your latest local changes to GitHub - this notebook clones the
repo fresh every time, so whatever's on `origin/main` is exactly what runs.

Code lives in Colab's local (ephemeral) disk via `git clone` - cheap to re-fetch every session.
Only the data (images/manifest) and checkpoints live on Drive, so they persist across sessions
without the code itself needing to.

**To resume in a later session:** push any code changes, then just re-run every cell top to
bottom. Manifest building is skipped automatically if `manifest.csv` already exists on Drive,
and training resumes from the last Drive checkpoint automatically.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU - go enable the T4 GPU runtime')
assert torch.cuda.is_available(), 'Enable a GPU runtime: Runtime -> Change runtime type -> T4 GPU'

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/hindh00/deepfake-image-detector.git'

!rm -rf /content/repo
!git clone $REPO_URL /content/repo

PROJECT_DIR = Path('/content/repo/ml')
assert PROJECT_DIR.exists()

In [ ]:
!pip install -q -r {PROJECT_DIR}/requirements.txt

In [ ]:
import os
from google.colab import userdata

# Add HF_TOKEN under Colab's Secrets tab (key icon, left sidebar) first - never paste
# the raw token into a cell. Same free read-only token from huggingface.co/settings/tokens.
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_DIR))

from src.datasets.build_manifest import build_manifest
from src.train import load_configs, train

In [ ]:
# Data and checkpoints both live on Drive - persist across sessions even though the
# cloned code (above) is re-fetched fresh every time.
DRIVE_ROOT = Path('/content/drive/MyDrive/deepfake-image-detector')
DATA_DIR = DRIVE_ROOT / 'data'
CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints'
DATA_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# First real validation run: ~300 images per source (~1.5-2k total across all 5 sources,
# split train/val/test + the genimage_pp held-out set). Skipped automatically on later
# sessions once manifest.csv already exists on Drive - won't re-download or re-sample.
# Kept modest (vs. 800) to stay under free-tier Colab's ~12.7GB RAM ceiling; raise it
# once you've confirmed a run completes cleanly.
PER_SOURCE_CAP = 300
MANIFEST_PATH = DATA_DIR / 'manifest.csv'

if not MANIFEST_PATH.exists():
    rows = build_manifest(per_source_cap=PER_SOURCE_CAP, data_dir=DATA_DIR)
    print(f'built manifest: {len(rows)} rows -> {MANIFEST_PATH}')
else:
    print(f'manifest already exists at {MANIFEST_PATH}, skipping build')

**If this was the first run (manifest just got built):** `Runtime -> Restart session`, then
re-run cells 1-7 (quick - just mounting Drive, checking the GPU, cloning, installing, and
pointing at the paths) before running the training cell below. Streaming+decoding images
from HF and then loading the model/training stack in the *same* session stacks both sets of
memory on top of each other - a restart drops the manifest-building memory before training
starts, and `manifest.csv` already exists on Drive so it won't be rebuilt.

In [ ]:
model_cfg, train_cfg = load_configs()

result = train(model_cfg, train_cfg, checkpoint_dir=CHECKPOINT_DIR, data_dir=DATA_DIR)
print('best val loss:', result['best_val_loss'])

Check progress anytime (even from another session) by opening, in Drive under
`deepfake-image-detector/checkpoints/`:
- `training_curves.png` - train/val loss, per-branch loss, gate weights, accuracy
- `canary/epoch_*.png` - explainability snapshots on the same fixed batch each epoch
- `training_history.json` - raw per-epoch numbers